In [ ]:
import os
import re
import math
import time
from pathlib import Path
from functools import partial

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import widgets
from scipy.spatial import ConvexHull
import mcubes
import quadprog

from pydrake.all import (
    Parser, RigidTransform, RotationMatrix, RollPitchYaw, Role,
    CspaceFreePolytope, SeparatingPlaneOrder, Parallelism, RandomGenerator,
    RobotDiagramBuilder, SceneGraphCollisionChecker,
    IrisFromCliqueCoverOptions, IrisInConfigurationSpaceFromCliqueCover,
    IrisZoOptions, IrisNp2Options, CommonSampledIrisOptions,
)
from pydrake.geometry.optimization import HPolyhedron, VPolytope, Point, Hyperellipsoid
from pydrake.solvers import MathematicalProgram, Solve, MosekSolver
from pydrake.multibody.inverse_kinematics import InverseKinematics

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "summer-work" else NOTEBOOK_DIR
MY_SDFS = REPO_ROOT / "my_sdfs"

import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from ciris_plant_visualizer import CIrisPlantVisualizer

full_joint_names = [
    "panda_joint1", "panda_joint2", "panda_joint3", "panda_joint4", "panda_joint5", "panda_joint6",
    "panda_joint7", "panda_finger_joint1", "panda_finger_joint2", "cap_to_base",
]
q_grasp_full = np.array([
    -1.28718907e+00, 1.31290357e+00, 1.18527863e+00, -2.21273568e+00,
    -1.58448521e+00, 2.03056033e+00, 1.66303822e+00, -2.40000000e-02,
    2.40000000e-02, 8.88178420e-16,
])
q_grasp_by_name = dict(zip(full_joint_names, q_grasp_full))

free_joint_names = ["panda_joint7", "panda_finger_joint1", "cap_to_base"]
locked_arm_joint_names = [f"panda_joint{i}" for i in range(1, 7)] + ["panda_finger_joint2"]
q_free_grasp = np.array([q_grasp_by_name[name] for name in free_joint_names])
print("free_joint_names:", free_joint_names)
print("locked_arm_joint_names:", locked_arm_joint_names)
print("q_free_grasp:", q_free_grasp)

In [ ]:
def Rx(a):
    c, s = math.cos(a), math.sin(a)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])

def Ry(a):
    c, s = math.cos(a), math.sin(a)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])

def Rz(a):
    c, s = math.cos(a), math.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

def rpy_to_R(roll, pitch, yaw):
    return Rz(yaw) @ Ry(pitch) @ Rx(roll)

def R_to_rpy(R):
    pitch = math.atan2(-R[2, 0], math.sqrt(R[0, 0] ** 2 + R[1, 0] ** 2))
    roll = math.atan2(R[2, 1], R[2, 2])
    yaw = math.atan2(R[1, 0], R[0, 0])
    return roll, pitch, yaw

def fmt(x):
    return "0" if abs(x) < 5e-13 else f"{x:.12g}"

def write_locked_panda_arm_urdf(input_path=MY_SDFS / "panda_arm.urdf"):
    text = Path(input_path).read_text()
    for joint_name in [f"panda_joint{i}" for i in range(1, 7)]:
        q = q_grasp_by_name[joint_name]
        pattern = re.compile(rf'<joint name="{joint_name}" type="revolute">.*?</joint>', re.S)
        match = pattern.search(text)
        if match is None:
            raise RuntimeError(f"Could not find revolute block for {joint_name}")
        block = match.group(0)
        origin = re.search(r'<origin rpy="([^"]+)" xyz="([^"]+)"/>', block)
        parent = re.search(r'<parent link="([^"]+)"/>', block)
        child = re.search(r'<child link="([^"]+)"/>', block)
        rpy0 = [float(v) for v in origin.group(1).split()]
        R_fixed = rpy_to_R(*rpy0) @ Rz(q)
        rpy_fixed = " ".join(fmt(v) for v in R_to_rpy(R_fixed))
        new_block = (
            f'<joint name="{joint_name}" type="fixed">\n'
            f'    <!-- Locked at q_grasp[{joint_name}] = {q:.8g} rad. -->\n'
            f'    <origin rpy="{rpy_fixed}" xyz="{origin.group(2)}"/>\n'
            f'    <parent link="{parent.group(1)}"/>\n'
            f'    <child link="{child.group(1)}"/>\n'
            f'  </joint>'
        )
        text = text[:match.start()] + new_block + text[match.end():]
    output_path = MY_SDFS / "panda_arm_locked_from_q_grasp.urdf"
    output_path.write_text(text)
    return output_path

locked_arm_urdf = write_locked_panda_arm_urdf()
print("Wrote", locked_arm_urdf)

In [ ]:
print("Setting up the reduced 3-DoF plant...")
builder = RobotDiagramBuilder(time_step=0.0)
plant = builder.plant()
scene_graph = builder.scene_graph()
parser = Parser(plant, scene_graph)
parser.SetAutoRenaming(True)

panda_arm = parser.AddModels(file_name=str(locked_arm_urdf))[0]
panda_hand = parser.AddModels(file_name=str(MY_SDFS / "panda_hand.urdf"))[0]
cap = parser.AddModels(file_name=str(MY_SDFS / "bottle_cap.sdf"))[0]

plant.WeldFrames(plant.world_frame(), plant.GetFrameByName("panda_link0", panda_arm), RigidTransform())
X_8H = RigidTransform(RollPitchYaw(0.0, 0.0, -np.deg2rad(45.0)), [0.0, 0.0, 0.0])
plant.WeldFrames(plant.GetFrameByName("panda_link8", panda_arm), plant.GetFrameByName("panda_hand", panda_hand), X_8H)
plant.WeldFrames(plant.world_frame(), plant.GetFrameByName("base_link", cap), RigidTransform(RotationMatrix(), [0.5, 0, 0]))

plant.Finalize()
print("Number of positions:", plant.num_positions())
print("Position names:", plant.GetPositionNames())
assert plant.num_positions() == 3, plant.GetPositionNames()

q_star = q_free_grasp.copy()
cspace_free_polytope = CspaceFreePolytope(plant, scene_graph, SeparatingPlaneOrder.kAffine, q_star)
visualizer = CIrisPlantVisualizer(plant, builder, scene_graph, cspace_free_polytope, viz_role=Role.kIllustration)
plant_context = visualizer.plant_context
diagram = visualizer.task_space_diagram
diagram_context = visualizer.task_space_diagram_context
plant.SetPositions(plant_context, q_star)
diagram.ForcedPublish(diagram_context)
print("Initial q_star set to:", q_star)

In [ ]:
def joint_position_index(plant, joint_name, model_instance):
    joint = plant.GetJointByName(joint_name, model_instance)
    if joint.num_positions() != 1:
        raise ValueError(f"{joint_name} is not a free 1-DoF joint")
    return joint.position_start()

active_indices = [
    joint_position_index(plant, "panda_joint7", panda_arm),
    joint_position_index(plant, "panda_finger_joint1", panda_hand),
    joint_position_index(plant, "cap_to_base", cap),
]
q_grasp = q_star.copy()
print("Active indices:", active_indices)
for i, name in enumerate(plant.GetPositionNames()):
    print(f"{i}: {name}")

model_instances = [
    plant.GetModelInstanceByName("panda"),
    plant.GetModelInstanceByName("panda_hand"),
    plant.GetModelInstanceByName("bottle_cap"),
]
generator = RandomGenerator(1234)
checker = SceneGraphCollisionChecker(model=diagram, robot_model_instances=model_instances, edge_step_size=0.01)

common = CommonSampledIrisOptions()
common.delta = 0.05
common.epsilon = 0.01
common.max_iterations = 10
common.parallelism = Parallelism(32)
common.verbose = True
common.configuration_space_margin = 1e-5
common.termination_threshold = 1e-5
common.relative_termination_threshold = 1e-4
common.remove_all_collisions_possible = False

iris_zo_options = IrisZoOptions()
iris_zo_options.bisection_steps = 30
iris_zo_options.sampled_iris_options = common

options = IrisFromCliqueCoverOptions()
options.num_points_per_visibility_round = 200
options.coverage_termination_threshold = 0.9
options.parallelism = common.parallelism
options.iris_options = iris_zo_options

# Uncomment to run the reduced 3D IRIS computation.
# regions = IrisInConfigurationSpaceFromCliqueCover(checker, options, generator, [])
# print("number of regions:", len(regions))